In [ ]:
import os
from bs4 import BeautifulSoup
import requests
import time
from datetime import datetime
import json
from zipfile import ZipFile
import pandas as pd
import shutil
import zipfile
import random

### Define PATH constant

In [ ]:
BASE_PATH = "data"
HTML_PATH = BASE_PATH + "/html"
USER_PATH = BASE_PATH + "/users"
ACCESS_TOKEN = "insira_seu_token"
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
ANIME_PATH = "data/animes"


# 1. Get Clubs IDs

In [ ]:
if not os.path.exists(f"{BASE_PATH}/clubs.txt"):
    with open(f"{BASE_PATH}/clubs.txt", "w", encoding="UTF-8") as file:
        pass

In [ ]:
def get_number(string):
    return int(string.strip().replace(",", ""))


clubs_id = set()
possibles_users = 0
page = 1

while True:
    print(f"\r{page}", end="")

    time.sleep(3)  # Wait 3 seconds per page
    data = requests.get(f"https://myanimelist.net/clubs.php?p={page}")
    soup = BeautifulSoup(data.text, "html.parser")
    rows = soup.find_all("tr", {"class": "table-data"})
    for row in rows:
        members = get_number(row.find("td", {"class": "ac"}).text)
        club_id = get_number(
            row.find("a", {"class": "fw-b"}).get("href").split("=")[-1]
        )
        if (
            club_id not in clubs_id and members > 30
        ):  # Only save groups with more than 100 members
            possibles_users += members
            clubs_id.add(club_id)

    page += 1
    if possibles_users > 10000:  # Threshold to stop
        break

with open(f"{BASE_PATH}/clubs.txt", "w") as file:
    for club in clubs_id:
        file.write(f"{club}\n")

# 2. Get usernames in every clubs

In [ ]:
if not os.path.exists(f"{BASE_PATH}/users_list.txt"):
    with open(f"{BASE_PATH}/users_list.txt", "w", encoding="UTF-8") as file:
        pass
    
if not os.path.exists(f"{BASE_PATH}/_revised_clubs.txt"):
    with open(f"{BASE_PATH}/_revised_clubs.txt", "w", encoding="UTF-8") as file:
        pass

In [ ]:
with open(f"{BASE_PATH}/clubs.txt") as file:
    clubs_id = [x.strip() for x in file.readlines()]

with open(f"{BASE_PATH}/users_list.txt", encoding="UTF-8") as file:
    users = set([x.strip() for x in file.readlines()])

with open(f"{BASE_PATH}/_revised_clubs.txt", encoding="UTF-8") as file:
    revised_clubs = set([int(x.strip()) for x in file.readlines()])

len(users), len(revised_clubs), len(clubs_id)

In [ ]:
%load_ext jupyternotify

In [ ]:
for i, club_id in enumerate(clubs_id):
    if club_id in revised_clubs:
        continue

    page = 1
    while True:
        print(f"\r{i+1}/{len(clubs_id)} --> {str(page).zfill(2)}", end="")
        link = f"https://api.jikan.moe/v4/clubs/{club_id}/members?page={page}"

        try:
            time.sleep(4.2)
            resp = requests.get(link)
        except KeyboardInterrupt:
            raise
        except Exception as e:
            # On network error or unexpected error, wait and retry
            time.sleep(120)
            continue

        if resp.status_code != 200:
            break

        payload = resp.json()
        members = payload.get("data", [])
        if not members:
            break

        with open(f"{BASE_PATH}/users_list.txt", "a", encoding="utf-8") as file:
            for user in members:
                username = user.get("username")
                if username and username not in users:
                    file.write(f"{username}\n")
                    users.add(username)

        # Check if there is a next page
        pagination = payload.get("pagination", {})
        has_next = pagination.get("has_next_page", False)
        if not has_next:
            break

        page += 1

    revised_clubs.add(club_id)
    with open(f"{BASE_PATH}/_revised_clubs.txt", "a", encoding="utf-8") as file:
        file.write(f"{club_id}\n")


In [ ]:
with open(f"{BASE_PATH}/users_list.txt", encoding="UTF-8") as file:
    users = list(set([x.strip() for x in file.readlines()]))[1:]
    random.shuffle(users)

with open(f"{BASE_PATH}/users.csv", "w", encoding="UTF-8") as file:
    file.write("user_id,username\n")
    for i, user in enumerate(users):
        file.write(f"{i},{user}\n")

# 3. Get animelist per user

In [ ]:
with open(f"{BASE_PATH}/users.csv", "r", encoding="UTF-8") as file:
    file.readline()
    users = [x.strip().split(",") for x in file.readlines()]
    users = [(int(x[0]), x[1]) for x in users]

last_revised_users = -1
if os.path.exists(f"{BASE_PATH}/_last_revised_users.txt"):
    with open(f"{BASE_PATH}/_last_revised_users.txt", "r", encoding="UTF-8") as file:
        last_revised_users = int(file.readline())

len(users), last_revised_users

In [ ]:
%reload_ext jupyternotify

In [ ]:
def get_user_animelist(username):
    all_animes = []
    offset = 0
    limit = 100

    while True:
        url = (
            f"https://api.myanimelist.net/v2/users/{username}/animelist"
            f"?fields=list_status,num_episodes&limit={limit}&offset={offset}"
        )

        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
        except KeyboardInterrupt:
            raise
        except Exception as e:
            print(f"Error: {e}, retrying in 120s...")
            time.sleep(120)
            continue

        if resp.status_code == 401:
            raise Exception("Access token expired or invalid.")
        elif resp.status_code != 200:
            print(f"Request failed ({resp.status_code}): {resp.text}")
            break

        payload = resp.json()
        data = payload.get("data", [])
        if not data:
            break

        for item in data:
            anime = item["node"]
            anime_id = anime["id"]
            list_status = item.get("list_status", {})
            score = list_status.get("score", 0)
            watching_status = list_status.get("status", "")
            watched_episodes = list_status.get("num_episodes_watched", 0)
            all_animes.append((anime_id, score, watching_status, watched_episodes))

        # Pagination
        if "paging" in payload and "next" in payload["paging"]:
            offset += limit
            time.sleep(2.5)
        else:
            break

    return all_animes


# === Main loop ===
for i, (user_id, username) in enumerate(users):
    if user_id <= last_revised_users:
        continue

    now = datetime.now()
    print(f'\r{str(now).split(".")[0]} --> {i+1}/{len(users)}', end="")

    all_animes = get_user_animelist(username)
    print(f"\nFetched {len(all_animes)} animes for {username}")

    if all_animes:
        user_file = f"{USER_PATH}/{user_id}.csv"
        with open(user_file, "w", encoding="utf-8") as f:
            f.write("anime_id,score,watching_status,watched_episodes\n")
            for anime_id, score, status, episodes in all_animes:
                f.write(f"{anime_id},{score},{status},{episodes}\n")
        print(f"Saved: {user_file}")

    revised_users = user_id
    with open(f"{BASE_PATH}/_last_revised_users.txt", "w", encoding="utf-8") as file:
        file.write(f"{user_id}\n")


# 4. Get unique anime

In [ ]:
unique_anime = set()
folder = os.listdir(USER_PATH)
for i, user_file in enumerate(folder):
    if ".csv" not in user_file:
        continue

    print(f"\r{i + 1}/{len(folder)}", end="")
    with open(f"{USER_PATH}/{user_file}", "r") as file:
        file.readline()
        for line in file:
            anime = line.strip().split(",")[0]
            unique_anime.add(anime)

print("         ")
print(len(unique_anime))

In [ ]:
if not os.path.exists(f"{BASE_PATH}/_revised_animes.txt"):
    with open(f"{BASE_PATH}/_revised_animes.txt", "w", encoding="UTF-8") as file:
        pass

In [ ]:
with open(f"{BASE_PATH}/_revised_animes.txt", encoding="UTF-8") as file:
    revised_animes = set([int(x.strip()) for x in file.readlines()])

In [ ]:
unique_anime_int = {int(x) for x in unique_anime}


In [ ]:
len(revised_animes)

In [ ]:
%reload_ext jupyternotify

In [ ]:
for i, unique_id in enumerate(unique_anime):
    if int(unique_id) in revised_animes:
        #print(f"pulei {unique_id}")
        continue
    print(f"\r{len(revised_animes)}/{len(unique_anime)}", end="")
    link = f"https://api.jikan.moe/v4/anime/{unique_id}"

    try:
        time.sleep(4.2)
        resp = requests.get(link)
    except KeyboardInterrupt:
        raise
    except Exception as e:
        # On network error or unexpected error, wait and retry
        time.sleep(120)
        continue
    
    if resp.status_code != 200:
        break

    try:
        # Parse JSON
        payload = resp.json()

        # Save JSON
        out_path = os.path.join(ANIME_PATH, f"{unique_id}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    except Exception as e:
        break

    revised_animes.add(int(unique_id))
    with open(f"{BASE_PATH}/_revised_animes.txt", "a", encoding="utf-8") as file:
        #print(f"escrevi {unique_id}")
        file.write(f"{unique_id}\n")

# 5. Create animes.csv

In [ ]:
all_animes = []

for filename in os.listdir(ANIME_PATH):
    if filename.endswith(".json"):
        filepath = os.path.join(ANIME_PATH, filename)
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
                if "data" in data:
                    all_animes.append(data["data"])
                else:
                    all_animes.append(data)
        except Exception as e:
            print(f"Error loading {filename}: {e}")

In [ ]:
anime_data = pd.json_normalize(all_animes)

In [ ]:
anime_data.columns

In [ ]:
anime_data = anime_data[['mal_id', 'title', 'score', 'members', 'status', 'aired.string']]

In [ ]:
anime_data['score'].fillna(value = 0, inplace=True)

In [ ]:
anime_data = anime_data.dropna()

In [ ]:
anime_data.isna().sum()

In [ ]:
anime_data.to_csv("data/animes.csv", index=False, encoding="utf-8")

# 6. Create anime_list.csv

In [ ]:
import re

In [ ]:
all_anime_lists = []

for filename in os.listdir(USER_PATH):
    if filename.endswith(".csv"):
        filepath = os.path.join(USER_PATH, filename)
        user_id = re.findall(r"\d+", filename)
        user_id = user_id[0] if user_id else None
        try:
            data = pd.read_csv(filepath)
            data["user_id"] = int(user_id)
            all_anime_lists.append(data)
        except Exception as e:
            print(f"Error reading {filename}: {e}")

In [ ]:
anime_list = pd.concat(all_anime_lists, ignore_index=True)

In [ ]:
anime_list.drop('watching_status', axis=1, inplace=True)

In [ ]:
anime_list.rename(columns={"score":"rating"}, inplace=True)

In [ ]:
anime_list.isna().sum()

In [ ]:
unified_anime_list = anime_list.merge(
    anime_data,
    left_on="anime_id",   # ID from user list
    right_on="mal_id",    # ID from metadata
    how="left"            # keep ALL user rows
)


In [ ]:
unified_anime_list.sort_values(by='user_id', ascending=True, inplace=True)

In [ ]:
unified_anime_list

In [ ]:
unified_anime_list = unified_anime_list.dropna()

In [ ]:
unified_anime_list.to_csv("data/anime_list.csv", index=False, encoding="utf-8")